---
# IMPORTS

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import pyarrow
import config
import password

from src.data_loading import load_crsp, load_futures, load_crsp_polars, wrds_fetch, load_cz_monthly, Fama_French_fetch
from src.preprocessing import clean_crsp, clean_futures
from src.feature_engineering import add_target, add_volatility_momentum, crosssectional_rank, get_feature_cols, polars_features
from src.utils import generate_batches, train_val_test_split, split_batch, shrink

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)

---
# Load daily dataset and Feature engineering

In [2]:
from src.feature_engineering import polars_features

data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(f'Dataset RAM size before shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
data = shrink(data)
print(f'Dataset RAM size after shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
print(data.dtypes)
print(data.head())



(19603583, 4)
['PERMNO', 'date', 'ret', 'mkt_ret']
Dataset RAM size before shrink : 0.51Gb
Dataset RAM size after shrink : 0.37Gb
PERMNO              int32
date       datetime64[ms]
ret               float32
mkt_ret           float32
dtype: object
   PERMNO       date     ret  mkt_ret
0   10001 2000-01-03  0.0074  -0.0095
1   10001 2000-01-04 -0.0146  -0.0383
2   10001 2000-01-05  0.0148   0.0019
3   10001 2000-01-06 -0.0073   0.0010
4   10001 2000-01-07 -0.0074   0.0271


In [3]:
df = polars_features(data, value_col='ret', date_col='date', target_col = 'ret', reversal=True)

df = df.to_pandas()
 
df.head()

,PERMNO,date,ret,mkt_ret,reversal_1d,target,mom_5d,mom_21d,mom_63d,mom_126d,mom_252d,vol_21d,vol_63d,vol_252d,vol_w_mom_5d_std_5d,vol_w_mom_21d_std_21d,vol_w_mom_63d_std_63d,vol_w_mom_126d_std_126d,vol_w_mom_252d_std_252d
0,10001,2000-01-03,0.0074,-0.0095,NaN,0.0074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,2000-01-04,-0.0146,-0.0383,0.0073,-0.0146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10001,2000-01-05,0.0148,0.0019,-0.0147,0.0148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10001,2000-01-06,-0.0073,0.0010,0.0147,-0.0073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,2000-01-07,-0.0074,0.0271,-0.0073,-0.0074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.head(15)

,PERMNO,date,ret,mkt_ret,reversal_1d,mom_5d,mom_21d,mom_63d,mom_126d,mom_252d,vol_21d,vol_63d,vol_252d,vol_w_mom_5d_std_5d,vol_w_mom_21d_std_21d,vol_w_mom_63d_std_63d,vol_w_mom_126d_std_126d,vol_w_mom_252d_std_252d
0,10001,2000-01-03,0.0074,-0.0095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,2000-01-04,-0.0146,-0.0383,0.0073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10001,2000-01-05,0.0148,0.0019,-0.0147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10001,2000-01-06,-0.0073,0.0010,0.0147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,2000-01-07,-0.0074,0.0271,-0.0073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,10001,2000-01-10,0.0000,0.0112,-0.0074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,10001,2000-01-11,0.0037,-0.0131,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,10001,2000-01-12,-0.0111,-0.0044,0.0037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,10001,2000-01-13,-0.0299,0.0122,-0.0111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10001,2000-01-14,0.0308,0.0107,-0.0303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
from src.feature_engineering import build_features

"""

To run once, DO NOT RUN if features.parquet already exists in /data/processed

"""

# 1. Load data and shrink

data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(f'Dataset RAM size before shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
data = shrink(data)
print(f'Dataset RAM size after shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
print(data.dtypes)
print(data.head())


"""

build_features function creates the following :

- 1 day reversal
- Momentum on multiple windows  
- Momentum scaled by volatility on multiple windows
- Volatility on multiple windows

"""

# 2. Build features

df = build_features(data, config.VALUE_RETURN, trading_interval=False)
 
print(df.head())


# 3. Shrink and upload to .parquet

print(f'Dataset RAM size before shrink : {df.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
df = shrink(df)
print(f'Dataset RAM size after shrink : {df.memory_usage(index=True).sum() / 1024**3:.2f}Gb')


df.to_parquet(config.FEATURES_PATH_CLEAN, compression='zstd')

(19603583, 4)
['PERMNO', 'date', 'ret', 'mkt_ret']
Dataset RAM size before shrink : 0.51Gb
Dataset RAM size after shrink : 0.37Gb
PERMNO              int32
date       datetime64[ms]
ret               float32
mkt_ret           float32
dtype: object
   PERMNO       date     ret  mkt_ret
0   10001 2000-01-03  0.0074  -0.0095
1   10001 2000-01-04 -0.0146  -0.0383
2   10001 2000-01-05  0.0148   0.0019
3   10001 2000-01-06 -0.0073   0.0010
4   10001 2000-01-07 -0.0074   0.0271


/Users/dariofrey/Documents/Dario/Université/EPFL/MFE/MA-2/Machine Learning in Finance/ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/dariofrey/Documents/Dario/Université/EPFL/MFE/MA-2/Machine Learning in Finance/ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


     PERMNO       date     ret  mkt_ret  reversal_1d  mom_scaled_5d  mom_5d  \
201   10001 2000-10-18 -0.0071  -0.0058       0.0478        -0.1288 -0.0149   
202   10001 2000-10-19  0.0000   0.0347      -0.0238         0.2008  0.1983   
203   10001 2000-10-20  0.0071   0.0059      -0.1657        -0.0070 -0.0096   
204   10001 2000-10-23 -0.0071  -0.0008       0.0797         0.1743  0.0773   
205   10001 2000-10-24  0.0143   0.0017      -0.1373        -0.3645 -0.1235   

     vol_5d  mom_scaled_21d  mom_21d  ...  mom_scaled_63d  mom_63d  vol_63d  \
201 -0.2534          0.2205   0.2225  ...          0.3624   0.3400  -0.2376   
202 -0.4209          0.3127   0.3083  ...          0.3661   0.3450  -0.2372   
203 -0.4258          0.2253   0.2244  ...          0.3036   0.2897  -0.2471   
204 -0.4107          0.2316   0.2256  ...          0.3008   0.2812  -0.2474   
205 -0.4532          0.1132   0.1310  ...          0.3197   0.2889  -0.2476   

     mom_scaled_126d  mom_126d  vol_126d  mom_scal

---
# Load Chen-Zimmerman Dataset

In [ ]:
from src.data_loading import load_cz_monthly


# 1. Load and shrink

cz_daily = load_cz_monthly()
print(f'Dataset RAM size before shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
cz_daily = shrink(cz_daily)
print(f'Dataset RAM size after shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')


# 2. Index cz dataset on date

cz_indexed = cz_daily.set_index('date').sort_index()
cz_indexed.index = pd.to_datetime(cz_indexed.index).astype('datetime64[ms]') # Prepare to merge


# 3. Upload to .parquet 

cz_indexed.to_parquet(config.CZ_PATH_CLEAN, compression='zstd')
cz_indexed.head()

Initial Size of the dataset: (1140, 206)
Total Date range : 1926-01-30 00:00:00 -> 2020-12-31 00:00:00
Dataset shape after dropping column with less than 90.0% completion: (1140, 57)
Total column dropped so far : 149
Dataset shape after dropping highly correlated (corr_coef > 0.95) columns: (1140, 55)
Total column dropped so far : 151
Dataset RAM size before shrink : 0.01Gb
Dataset RAM size after shrink : 0.01Gb


,Beta,BetaFP,BetaTailRisk,BidAskSpread,CompEquIss,Coskewness,DivInit,DivOmit,DivSeason,DivYieldST,...,Size,Spinoff,std_turn,STreversal,VolMkt,VolSD,VolumeTrend,zerotrade,zerotradeAlt1,zerotradeAlt12
date,,,,,,,,,,,,,,,,,,,,,
1932-01-30,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-01-31,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-01,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-02,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-03,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856


---
# Generate Batch and Merge

Generating the random sampled batch and merging the other dataset on the batch. 

We merge other dataset with the daily dataset only at the batch level in order to largely decrease the RAM usage duriung the operation considering the size of the daily dataframe with around 6700 stocks.

In [3]:
features = pd.read_parquet(config.FEATURES_PATH_CLEAN)
cz = pd.read_parquet(config.CZ_PATH_CLEAN)
print(f'Number of unique PERMNO in dataset : {features['PERMNO'].nunique()}')

split_batch_dict = split_batch(features, 'PERMNO', 'date', df_to_join=[cz], batch_number=config.BATCH_NUMBER, batch_size=config.BATCH_SIZE, overlap=False)

Number of unique PERMNO in dataset : 6647
Total unique dates: 6088
  train: 2000-10-17 → 2017-09-25  (4261 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6089
  train: 2000-10-16 → 2017-09-25  (4262 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6089
  train: 2000-10-16 → 2017-09-25  (4262 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6090
  train: 2000-10-16 → 2017-09-26  (4263 dates, 70.0%)
  val  : 2017-09-27 → 2021-05-13  (913 dates, 15.0%)
  test : 2021-05-14 → 2024-12-31  (914 dates, 15.0%)
Total unique dates: 6088
  train: 2000-10-17 → 2017-09-25  (4261 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)


---
# Fama-French 3 Factors

In [ ]:
# Start by loading the data
permno_list = features['PERMNO'].unique()


# Start and End date are taken as min and max of initial dataset as for now the ff3 is for benchmark purpose


ff3_df = Fama_French_fetch(permno_list = permno_list, start_date = features['date'].min(), end_date = features['date'].max())


/!| PLEASE FILL CREDENTIALS /!|
WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\dario\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


---
# Handling NaN due to merge/join

As we merged the different dataset, we use how='left' in order to use the daily dataset as the baseline. However, due to differences in data range among datasets, we will have to handle NaN after merging.

This is a Work In Progress !